# Register of Members’ Interests: Westminster and Senedd

This notebook looks at two JSON files prepared by MySociety. It first checks how each file is organised, then uses a few simple questions to explore the entries.

- [Westminster Commons RMFI dataset](https://pages.mysociety.org/parl_register_interests/datasets/commons_rmfi/latest): `rmfi_commons.json`
- [Senedd Register of Interests dataset](https://pages.mysociety.org/parl_register_interests/datasets/senedd_register_of_interest): `SRMI.json`

In [23]:
import json
import pandas as pd

## 1. Westminster: inspect the JSON structure

In [24]:
with open('rmfi_commons.json') as json_data:
   d = json.load(json_data)

commons_resources = d['resources']
commons_records = commons_resources[0]['data']

In [25]:
commons = pd.DataFrame(commons_records)
print('Rows and columns:', commons.shape)
commons.head(2)

Rows and columns: (4070, 9)


,id,parent_interest_id,member,party,mnis_id,twfy_id,category,category_code,summary
0,1049,,Ed Davey,liberal-democrat,188,10155,"Gifts, benefits and hospitality from UK sources",3,National Liberal Club - £798.00
1,1117,,Tim Farron,liberal-democrat,1591,11923,"Gifts, benefits and hospitality from UK sources",3,"National Liberal Club - £1,102.50"


The table has **4,070 rows and 9 columns** in this copy of the file. 

### Question 1: How many entries say “Property in London”, and how many MPs do they belong to?

In [26]:
property_rows = commons[commons['summary'] == 'Property in London']
member_counts = property_rows['member'].value_counts()

print('Matching register entries:', len(property_rows))
print('Different MPs:', property_rows['member'].nunique())
print('MPs with more than one matching entry:')
member_counts[member_counts > 1]

Matching register entries: 38
Different MPs: 35
MPs with more than one matching entry:


member
Jessica Toale    2
Jas Athwal       2
Rupert Lowe      2
Name: count, dtype: int64

**Answer:** There are **38 entries** for **35 MPs**. Jessica Toale, Jas Athwal, and Rupert Lowe each have two matching entries. The number of entries is higher than the number of MPs because some MPs appear more than once.

### Question 2: Which MPs have the most recorded visits outside the UK?

In [27]:
for resource in commons_resources:
    if resource['name'] == 'category_4':
        visits = pd.DataFrame(resource['data'])

print('Visit entries:', len(visits))
visits['member'].value_counts().head(10)

Visit entries: 385


member
Tom Tugendhat        14
Jeremy Corbyn        11
Naz Shah              8
Andrew Rosindell      8
George Freeman        7
Nigel Farage          6
Mr Richard Holden     5
Bill Esterson         5
Alicia Kearns         5
Priti Patel           5
Name: count, dtype: int64

**Answer:** In these **385 visit entries**, Tom Tugendhat has the most, with **14**. Jeremy Corbyn has **11**. Naz Shah and Andrew Rosindell have **8 each**. These are counts of entries in the file.

### Question 3: Which visit entries have the highest total declared support?

The visit records can have up to five donor value fields. The code adds the values in each row, treating blank values as zero, then sorts the rows from highest to lowest.

In [28]:
value_columns = [f'donors__value_{i}' for i in range(1, 6)]
visits[value_columns] = visits[value_columns].apply(pd.to_numeric, errors='coerce').fillna(0)
visits['declared_support'] = visits[value_columns].sum(axis=1).round(2)

top_visits = visits.sort_values('declared_support', ascending=False)
top_visits[['member', 'visit_locations__country_1', 'declared_support']].head(5)

,member,visit_locations__country_1,declared_support
334,Nigel Farage,United States,54921.02
264,Nigel Farage,Switzerland,51603.99
260,Al Carns,Nepal,47442.18
358,Mrs Kemi Badenoch,United States,27479.96
230,Nigel Farage,Maldives,25000.00


**Answer:** The largest total in one visit record is **£54,921.02** for Nigel Farage’s visit entry listing the United States. The next two are **£51,603.99** for a Nigel Farage entry listing Switzerland, and **£47,442.18** for an Al Carns entry listing Nepal. These totals are sums of the donor value fields in each record.

## 2. Senedd: inspect the JSON structure

The Senedd file has a similar outer structure: a top-level dictionary containing a `resources` list. In this file, that list has one resource, which contains the entries.

In [29]:
with open('SRMI.json') as json_data:
    d = json.load(json_data)

senedd_resources = d['resources']
senedd_records = senedd_resources[0]['data']

This resource has **1,755 records**. Each record is a dictionary with fields including the member’s name, the category, text describing the interest, and whether the entry is marked as being in the latest register.

In [30]:
senedd = pd.DataFrame(senedd_records)
print('Rows and columns:', senedd.shape)
senedd[['person_name', 'category_name', 'in_latest_register']].head(3)

Rows and columns: (1755, 11)


,person_name,category_name,in_latest_register
0,Paul Rock MS,Standing Order 5A: Voluntary record of Members...,True
1,Niamh Salkeld MS,Standing Order 5A: Voluntary record of Members...,True
2,Niamh Salkeld MS,Recording of employment of family members of o...,True


### Question 4: Which Senedd categories have the most entries in this file?

In [31]:
senedd['category_name'].value_counts().head(10)

category_name
Membership/Chairmanship of bodies in receipt of Senedd funds                                                      347
Standing Order 5A: Voluntary record of Members’ other interests                                                   316
Remunerated Employment, Directorships etc.                                                                        181
Gifts, hospitality, material benefit or advantage                                                                 177
Record of membership of societies                                                                                 151
Remunerated employment, office, profession etc                                                                    121
Financial sponsorship                                                                                              86
Membership/Chairmanship of bodies in receipt of direct or indirect Senedd Commission or Welsh Government funds     70
Record of the employment of family members

**Answer:** The largest category is **Membership/Chairmanship of bodies in receipt of Senedd funds**, with **347 entries**. Next are **Standing Order 5A: Voluntary record of Members’ other interests** with **316**, and **Remunerated Employment, Directorships etc.** with **181**. These counts describe rows in the supplied file.